## 2.5 章节实践

通过本章的系统学习，我们掌握了 Polar 码的信道极化原理、Arikan 蝶形编码和 SC 逐次消除解码算法，并通过对比无编码 BPSK 的 BER 曲线理解了编码增益的含义。为了巩固所学知识，现提供以下实践练习：

**计算 Polar 码的编码增益**，运行 Polar R=1/2 与无编码 BPSK 的 BER 扫描，在 BER=10⁻³ 处标定编码增益，判断是否满足 ≥ 2 dB 的合格标准。

相关概念：

- 编码增益：在目标 BER 处，有编码相比无编码省下的 Eb/N0 (dB)
- Polar(256, 112) R=1/2 的码率使每符号能量降为 Es = 0.5 Eb，噪声方差 σ² = 1/(2R×Eb/N0)

要求：

1. 补全 BER 扫描循环，在 -4~10 dB 范围内遍历 SNR
2. 补全编码增益计算：在 BER=10⁻³ 处分别查找无编码和 Polar 编码所需的 Eb/N0
3. 判断编码增益是否 ≥ 2 dB

请开始你的实践，体验从理解到创造的完整开发过程。

In [ ]:
%%writefile coding_gain.py
import sys
sys.path.insert(0, "../src")
import numpy as np
from scipy.special import erfc
from nearlink_sdr.common.polar import PolarEncoder, get_polar_decoder

N, K = 256, 112
enc = PolarEncoder(N, K)
dec = get_polar_decoder(N, K)
rng = np.random.default_rng(42)
snr_range = np.arange(-4, 11, 1)

# ---- 无编码 BPSK 理论 BER ----
ber_theory = [0.5 * erfc(np.sqrt(10**(s/10))) for s in snr_range]

# ---- Polar R=1/2 BER 扫描 ----
ber_coded = []
n_blocks = max(1, 5000 // K)
for snr in snr_range:
    total_err, total_bits = 0, 0
    R_val = K / N
    snr_linear = 10.0 ** (float(snr) / 10.0)
    # TODO: 补全噪声方差计算 (注意码率折算)
    noise_var = ???
    for _ in range(n_blocks):
        info = rng.integers(0, 2, size=K, dtype=np.int8)
        coded = enc.encode(info)
        tx = 1.0 - 2.0 * coded.astype(np.float64)
        noise = rng.normal(0, np.sqrt(noise_var), size=N)
        rx = tx + noise
        llr = 2.0 * rx / noise_var
        decoded = dec.decode(llr)
        total_err += int(np.sum(decoded != info))
        total_bits += K
    ber_coded.append(total_err / total_bits if total_bits > 0 else 0.0)
    print(f"SNR={snr:3.0f} dB  BER={ber_coded[-1]:.6f}")

# ---- 在 BER=1e-3 处计算编码增益 ----
snr_uncoded_1e3 = snr_range[np.argmin(np.abs(np.array(ber_theory) - 1e-3))]
snr_polar_1e3 = snr_range[np.argmin(np.abs(np.array(ber_coded) - 1e-3))]
# TODO: 补全编码增益计算公式
coding_gain = ???
print(f"\n编码增益 @ BER=1e-3: {coding_gain:.1f} dB")
print(f"合格 (>=2 dB): {'Yes' if coding_gain >= 2.0 else 'No'}")

执行以下命令进行编译并验证结果：


In [ ]:
!python coding_gain.py


执行以下代码获取答案

In [ ]:
!cat answer/coding_gain_answer.py